In [1]:
# Install required packages
!pip install optuna joblib kaggle_environments -q

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import optuna
import csv
import json
import math
import importlib
from joblib import Parallel, delayed
from kaggle_environments import make

# ─── 1. Setup Persistent Storage Paths ───────────────────────────────────────
# Change this folder name if you want it saved somewhere else in your Drive
BASE_DIR = '/content/drive/MyDrive/OrbitWars_Optimization'
os.makedirs(BASE_DIR, exist_ok=True)

# The SQLite database where Optuna will save every trial automatically
DB_PATH = os.path.join(BASE_DIR, 'optimization.db')
STORAGE_URL = f"sqlite:///{DB_PATH}"

# CSV Log paths for manual analysis later
LOG_CSV_PATH_V8 = os.path.join(BASE_DIR, 'v8_results_log.csv')
LOG_CSV_PATH_V10 = os.path.join(BASE_DIR, 'v10_results_log.csv')

print(f"✅ Environment initialized!")
print(f"📂 Data will be saved to: {BASE_DIR}")
print(f"🗄️ SQL Database URL: {STORAGE_URL}")

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 31.


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:Successfully loaded OpenSpiel environments: 31.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:   open_spiel_amazons


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon


...

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_python_repeated_pokerkit


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:   open_spiel_python_repeated_pokerkit


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: OpenSpiel games skipped: 0.


INFO:kaggle_environments.envs.open_spiel_env.open_spiel_env:OpenSpiel games skipped: 0.


✅ Environment initialized!
📂 Data will be saved to: /content/drive/MyDrive/OrbitWars_Optimization
🗄️ SQL Database URL: sqlite:////content/drive/MyDrive/OrbitWars_Optimization/optimization.db


In [3]:
# ─── 2. Initialize CSV Logs ──────────────────────────────────────────────────
def init_csv(path, params_list):
    if not os.path.exists(path):
        with open(path, mode='w', newline='') as f:
            writer = csv.writer(f)
            # Standard headers + the specific hyperparameters being tested
            headers = ['Trial', 'Fitness', 'WinRate', 'AvgScore', 'AvgTurns'] + params_list
            writer.writerow(headers)

# Initialize headers for V8 and V10
init_csv(LOG_CSV_PATH_V8, ['min_garrison', 'safety_margin_ships', 'min_launch_size'])
init_csv(LOG_CSV_PATH_V10, ['min_garrison', 'safety_margin_ships', 'min_launch_size',
                            'sun_safe_margin', 'production_roi_weight', 'en_route_buffer'])

# ─── 3. Define Global Benchmarking Seeds ─────────────────────────────────────
# We use fixed, hardcoded ranges to ensure zero variance between trials.
# 80 seeds for the optimizer to train on
TRAIN_SEEDS = list(range(1000, 1080))

# 30 unseen seeds for checking generalization later
TEST_SEEDS = list(range(2000, 2030))

# 150 seeds for the final V8 vs V10 showdown
SHOWDOWN_SEEDS = list(range(3000, 3150))

print(f"✅ CSV Logs initialized.")
print(f"🌱 Created Train ({len(TRAIN_SEEDS)}), Test ({len(TEST_SEEDS)}), and Showdown ({len(SHOWDOWN_SEEDS)}) seed sets.")

✅ CSV Logs initialized.
🌱 Created Train (80), Test (30), and Showdown (150) seed sets.


In [4]:
# ─── 1. Single Match Simulation Task ──────────────────────────────────────────
def simulate_single_game(seed, candidate_path, baseline_path, swap_seats=False):
    """
    Simulates a single game between two absolute paths.
    Maintains seat alternation to guarantee positional fairness.
    """
    try:
        # Create an isolated environment instance for this seed
        env = make("orbit_wars", configuration={"seed": seed, "episodeSteps": 500}, debug=False)

        # Handle seat rotation
        if not swap_seats:
            env.run([candidate_path, baseline_path])
            candidate_idx, baseline_idx = 0, 1
        else:
            env.run([baseline_path, candidate_path])
            candidate_idx, baseline_idx = 1, 0

        # Extract performance metrics from the final simulation state
        final_step = env.steps[-1]
        c_state = final_step[candidate_idx]
        b_state = final_step[baseline_idx]

        # Guard against None values in rewards
        c_reward = c_state.get('reward', 0) if c_state.get('reward') is not None else 0
        b_reward = b_state.get('reward', 0) if b_state.get('reward') is not None else 0

        turns = len(env.steps)

        # Categorize win condition status (higher score wins)
        if c_reward > b_reward:
            win_score = 1.0
        elif c_reward < b_reward:
            win_score = 0.0
        else:
            win_score = 0.5 # Standoff/Tie

        return {
            "win": win_score,
            "score_diff": c_reward - b_reward,
            "turns": turns,
            "error": False
        }

    except Exception as e:
        # Treat game crashes or unexpected timeouts as a catastrophic failure
        return {
            "win": 0.0,
            "score_diff": -100.0,
            "turns": 500,
            "error": True
        }

# ─── 2. Tournament Orchestrator ──────────────────────────────────────────────
def run_tournament(candidate_path, baseline_path, seeds, n_jobs=-1):
    """
    Executes a parallelized batch tournament over a fixed seed list.
    Calculates the normalized composite fitness score.
    """
    # Parallel map execution across CPU cores
    results = Parallel(n_jobs=n_jobs)(
        delayed(simulate_single_game)(seed, candidate_path, baseline_path, swap_seats=(i % 2 == 1))
        for i, seed in enumerate(seeds)
    )

    total_games = len(results)
    wins = sum(r['win'] for r in results)
    total_score_diff = sum(r['score_diff'] for r in results)
    total_turns = sum(r['turns'] for r in results)
    errors = sum(1 for r in results if r['error'])

    # Calculate performance averages
    win_rate = wins / total_games
    avg_score_diff = total_score_diff / total_games
    avg_turns = total_turns / total_games

    # Core Formula Implementation
    # Fitness = (WinRate * 1.0) + (AvgScoreDiff * 0.001) - (AvgTurns * 0.0001)
    fitness = (win_rate * 1.0) + (avg_score_diff * 0.001) - (avg_turns * 0.0001)

    if errors > 0:
        print(f"⚠️ Warning: {errors} game simulations threw internal runtime exceptions.")

    return fitness, win_rate, avg_score_diff, avg_turns

print("✅ Tournament Engine successfully initialized!")

✅ Tournament Engine successfully initialized!


In [5]:
# Verify absolute file system alignments within Google Drive
V7_PATH = os.path.join(BASE_DIR, 'agent_v7.py')
CURRENT_MAIN_PATH = os.path.join(BASE_DIR, 'v8.py')

# Quick layout assertions
assert os.path.exists(V7_PATH), f"❌ Crucial File Missing: agent_v7.py not detected at {V7_PATH}"
assert os.path.exists(CURRENT_MAIN_PATH), f"❌ Crucial File Missing: main.py not detected at {CURRENT_MAIN_PATH}"

print("🔄 Executing 80-game calibration matrix vs Static V7 Baseline...")
print("This may take 1-3 minutes depending on your allocated Colab CPU instance allocation...")

# Execute benchmark run
base_fitness, base_wr, base_score, base_turns = run_tournament(
    candidate_path=CURRENT_MAIN_PATH,
    baseline_path=V7_PATH,
    seeds=TRAIN_SEEDS
)

print("\n📊 ─────── THE CALIBRATION ZERO LINE ───────")
print(f"🏆 Base Fitness Score:  {base_fitness:.4f}")
print(f"📈 Win Rate vs V7:       {base_wr * 100:.2f}%")
print(f"🎯 Avg Score Delta:     {base_score:.2f}")
print(f"⏱️  Avg Game Duration:   {base_turns:.1f} turns")
print("────────────────────────────────────────────")
print("✅ Calibration complete. We are officially ready to initiate the optimization loop.")

🔄 Executing 80-game calibration matrix vs Static V7 Baseline...
This may take 1-3 minutes depending on your allocated Colab CPU instance allocation...

📊 ─────── THE CALIBRATION ZERO LINE ───────
🏆 Base Fitness Score:  0.3726
📈 Win Rate vs V7:       40.00%
🎯 Avg Score Delta:     -0.40
⏱️  Avg Game Duration:   269.6 turns
────────────────────────────────────────────
✅ Calibration complete. We are officially ready to initiate the optimization loop.


In [6]:
import re

# Set the official path for your unoptimized v8 source
V8_PATH = os.path.join(BASE_DIR, 'v8.py')
assert os.path.exists(V8_PATH), f"❌ Could not find your agent file at: {V8_PATH}"

# ─── 1. Define the Optuna Objective Function ──────────────────────────────────
def objective_v8(trial):
    """
    Optuna objective wrapper that suggests hyperparameter sets,
    injects them dynamically into a transient copy of v8.py,
    and runs a parallel tournament against V7 to compute fitness.
    """
    # Define search boundaries (Ranges & Constraints)
    min_garrison = trial.suggest_int('min_garrison', 2, 10)
    safety_margin_ships = trial.suggest_int('safety_margin_ships', 2, 10)
    min_launch_size = trial.suggest_int('min_launch_size', 5, 15)

    # Generate a unique transient filename to keep parallel core simulations isolated
    trial_agent_path = os.path.join(BASE_DIR, f"_v8_trial_{trial.number}.py")

    # Read original v8 source code
    with open(V8_PATH, 'r') as f:
        source_code = f.read()

    # Append overrides at the bottom of the script.
    # Python executes scripts sequentially, so these assignments override any global defaults at the top.
    override_block = f"""

# ─── OPTUNA DYNAMIC OVERRIDES ───
MIN_GARRISON = {min_garrison}
SAFETY_MARGIN_SHIPS = {safety_margin_ships}
MIN_LAUNCH_SIZE = {min_launch_size}
"""

    # Write the modified source code out to the temporary file
    with open(trial_agent_path, 'w') as f:
        f.write(source_code + override_block)

    try:
        # Run parallelized tournament on the training seeds
        fitness, win_rate, avg_score, avg_turns = run_tournament(
            candidate_path=trial_agent_path,
            baseline_path=V7_PATH,
            seeds=TRAIN_SEEDS,
            n_jobs=-1 # Utilize all available CPU cores on Colab
        )

        # Append results safely to the persistent CSV log
        with open(LOG_CSV_PATH_V8, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([trial.number, fitness, win_rate, avg_score, avg_turns,
                             min_garrison, safety_margin_ships, min_launch_size])

        return fitness

    finally:
        # Cleanup: Ensure temporary execution file is deleted even if execution crashes
        if os.path.exists(trial_agent_path):
            os.remove(trial_agent_path)

# ─── 2. Initialize and Launch the Optimization Study ─────────────────────────
print("🚀 Initializing Optuna study session...")

# Create or resume the study using your Drive SQL database
study_v8 = optuna.create_study(
    study_name="v8_tuning_arena",
    storage=STORAGE_URL,
    direction="maximize",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42) # Tree-structured Parzen Estimator for intelligent search
)

# Set up a target loop count (e.g., 50 trials for V8)
# If Colab disconnects, changing this number let's you extend the study later
TOTAL_TRIALS = 50

current_completed = len(study_v8.trials)
remaining_trials = max(0, TOTAL_TRIALS - current_completed)

if remaining_trials > 0:
    print(f"📈 Found {current_completed} existing trials. Commencing {remaining_trials} remaining trials...")
    study_v8.optimize(objective_v8, n_trials=remaining_trials)
else:
    print(f"🏁 Study already completed {current_completed} trials! Ready to check optimal settings.")

# ─── 3. Print the Best Discovered Parameters ──────────────────────────────────
print("\n🏆 ─────── V8 OPTIMIZATION COMPLETED ───────")
print(f"Best Trial Number:  {study_v8.best_trial.number}")
print(f"Maximized Fitness:  {study_v8.best_value:.4f}")
print("Optimal Configuration Parameters:")
for key, value in study_v8.best_params.items():
    print(f"  🔹 {key}: {value}")
print("────────────────────────────────────────────")

🚀 Initializing Optuna study session...


[I 2026-06-12 00:52:29,115] Using an existing study with name 'v8_tuning_arena' instead of creating a new one.


🏁 Study already completed 50 trials! Ready to check optimal settings.

🏆 ─────── V8 OPTIMIZATION COMPLETED ───────
Best Trial Number:  3
Maximized Fitness:  0.5600
Optimal Configuration Parameters:
  🔹 min_garrison: 8
  🔹 safety_margin_ships: 2
  🔹 min_launch_size: 15
────────────────────────────────────────────


In [7]:
import os
import csv
import optuna

# ─── 1. Verify V10 File Path Alignment ───────────────────────────────────────
V10_PATH = os.path.join(BASE_DIR, 'v10.py')
assert os.path.exists(V10_PATH), f"❌ Could not find your V10 agent file at: {V10_PATH}. Please ensure it is named v10.py"

# ─── 2. Define the V10 Optuna Objective Function ──────────────────────────────
def objective_v10(trial):
    """
    Optuna objective wrapper that suggests the expanded 6-parameter search space
    for V10, injects them into a temporary file, and runs the tournament vs V7.
    """
    # Define bounded search spaces based on our project plan constraints
    min_garrison = trial.suggest_int('min_garrison', 2, 6)
    safety_margin_ships = trial.suggest_int('safety_margin_ships', 2, 8)
    min_launch_size = trial.suggest_int('min_launch_size', 1, 5)

    # Continuous floating-point spaces for physics margins and ROI calculus
    sun_safe_margin = trial.suggest_float('sun_safe_margin', 1.0, 1.5)
    production_roi_weight = trial.suggest_float('production_roi_weight', 0.5, 2.0)

    # Core coordination buffer logic
    en_route_buffer = trial.suggest_int('en_route_buffer', 0, 5)

    # Generate an isolated temporary execution file path
    trial_agent_path = os.path.join(BASE_DIR, f"_v10_trial_{trial.number}.py")

    # Read original v10 production source code
    with open(V10_PATH, 'r') as f:
        source_code = f.read()

    # Compile the programmatic override injection block
    override_block = f"""

# ─── OPTUNA DYNAMIC OVERRIDES ───
MIN_GARRISON = {min_garrison}
SAFETY_MARGIN_SHIPS = {safety_margin_ships}
MIN_LAUNCH_SIZE = {min_launch_size}
SUN_SAFE_MARGIN = {sun_safe_margin:.4f}
PRODUCTION_ROI_WEIGHT = {production_roi_weight:.4f}
EN_ROUTE_BUFFER = {en_route_buffer}
"""

    # Write the modified source code out to the temporary runner file
    with open(trial_agent_path, 'w') as f:
        f.write(source_code + override_block)

    try:
        # Run parallelized tournament on the 80 training seeds
        fitness, win_rate, avg_score, avg_turns = run_tournament(
            candidate_path=trial_agent_path,
            baseline_path=V7_PATH,
            seeds=TRAIN_SEEDS,
            n_jobs=-1 # Dynamically scale across all allocated Colab CPU threads
        )

        # Append historical trial metrics safely to the persistent V10 CSV log
        with open(LOG_CSV_PATH_V10, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                trial.number, fitness, win_rate, avg_score, avg_turns,
                min_garrison, safety_margin_ships, min_launch_size,
                sun_safe_margin, production_roi_weight, en_route_buffer
            ])

        return fitness

    finally:
        # Cleanup: Hard purge the temporary runner to prevent disk clutter
        if os.path.exists(trial_agent_path):
            os.remove(trial_agent_path)

# ─── 3. Initialize and Launch the V10 Optimization Study ─────────────────────
print("🚀 Initializing V10 Optuna tuning session...")

study_v10 = optuna.create_study(
    study_name="v10_tuning_arena",
    storage=STORAGE_URL,
    direction="maximize",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=101) # Isolated random seed for the TPE geometric search
)

# Target search depth (6 parameters requires slightly more exploration than V8)
TOTAL_TRIALS_V10 = 60

current_completed_v10 = len(study_v10.trials)
remaining_trials_v10 = max(0, TOTAL_TRIALS_V10 - current_completed_v10)

if remaining_trials_v10 > 0:
    print(f"📈 Found {current_completed_v10} existing trials in SQLite database.")
    print(f"Commencing {remaining_trials_v10} remaining exploratory iterations...")
    study_v10.optimize(objective_v10, n_trials=remaining_trials_v10)
else:
    print(f"🏁 Study already completed {current_completed_v10} trials! Ready to analyze optimal V10 setup.")

# ─── 4. Output the Discovered Strategic Vector ───────────────────────────────
print("\n🏆 ─────── V10 OPTIMIZATION COMPLETED ───────")
print(f"Best Trial Number:  {study_v10.best_trial.number}")
print(f"Maximized Fitness:  {study_v10.best_value:.4f}")
print("Optimal Configuration Parameters:")
for key, value in study_v10.best_params.items():
    if isinstance(value, float):
        print(f"  🔹 {key}: {value:.4f}")
    else:
        print(f"  🔹 {key}: {value}")
print("────────────────────────────────────────────")

[I 2026-06-12 00:52:29,199] Using an existing study with name 'v10_tuning_arena' instead of creating a new one.


🚀 Initializing V10 Optuna tuning session...
🏁 Study already completed 60 trials! Ready to analyze optimal V10 setup.

🏆 ─────── V10 OPTIMIZATION COMPLETED ───────
Best Trial Number:  33
Maximized Fitness:  0.6140
Optimal Configuration Parameters:
  🔹 min_garrison: 6
  🔹 safety_margin_ships: 8
  🔹 min_launch_size: 1
  🔹 sun_safe_margin: 1.4641
  🔹 production_roi_weight: 0.8414
  🔹 en_route_buffer: 4
────────────────────────────────────────────


In [8]:
import os
from joblib import Parallel, delayed
from kaggle_environments import make

# ─── 1. Link the Optimized Files ─────────────────────────────────────────────
V8_OPT_PATH = os.path.join(BASE_DIR, 'v8_opt.py')
V10_OPT_PATH = os.path.join(BASE_DIR, 'v10_opt.py')

# Quick sanity check to ensure Colab sees the files you just uploaded
assert os.path.exists(V8_OPT_PATH), f"❌ Could not find {V8_OPT_PATH}"
assert os.path.exists(V10_OPT_PATH), f"❌ Could not find {V10_OPT_PATH}"

print("✅ Found optimized agents! V8 (Defensive) vs V10 (Aggressive).")

# ─── 2. Showdown Match Simulation Logic ──────────────────────────────────────
def simulate_showdown_match(seed, swap_seats=False):
    """
    Runs a direct match between Optimized V10 and Optimized V8.
    Accurately maps rewards back to the correct agent regardless of seat assignment.
    """
    try:
        # Create environment for this specific seed
        env = make("orbit_wars", configuration={"seed": seed, "episodeSteps": 500}, debug=False)

        if not swap_seats:
            # Player 1 (Index 0) = V10, Player 2 (Index 1) = V8
            env.run([V10_OPT_PATH, V8_OPT_PATH])
            v10_idx, v8_idx = 0, 1
        else:
            # Player 1 (Index 0) = V8, Player 2 (Index 1) = V10
            env.run([V8_OPT_PATH, V10_OPT_PATH])
            v8_idx, v10_idx = 0, 1

        # Extract the final results
        final_step = env.steps[-1]
        v10_reward = final_step[v10_idx].get('reward', 0) or 0
        v8_reward = final_step[v8_idx].get('reward', 0) or 0
        turns = len(env.steps)

        # Determine outcome based on score difference
        if v10_reward > v8_reward:
            return {"outcome": "V10_WIN", "turns": turns}
        elif v8_reward > v10_reward:
            return {"outcome": "V8_WIN", "turns": turns}
        else:
            return {"outcome": "DRAW", "turns": turns}

    except Exception as e:
        # If an execution error occurs (like a timeout), default it to a draw
        return {"outcome": "DRAW", "turns": 500}

# ─── 3. Run the Arena Tournament ─────────────────────────────────────────────
print(f"\n⚔️ Initiating the Grand Showdown over {len(SHOWDOWN_SEEDS)} unseen maps...")
print("Alternating player seating positions to ensure unbiased evaluation...")

# Execute parallel simulation across all Colab CPU cores
showdown_results = Parallel(n_jobs=-1)(
    delayed(simulate_showdown_match)(seed, swap_seats=(i % 2 == 1))
    for i, seed in enumerate(SHOWDOWN_SEEDS)
)

# ─── 4. Compile Metrics and Analytics ─────────────────────────────────────────
total_matches = len(showdown_results)
v10_wins = sum(1 for r in showdown_results if r['outcome'] == "V10_WIN")
v8_wins = sum(1 for r in showdown_results if r['outcome'] == "V8_WIN")
draws = sum(1 for r in showdown_results if r['outcome'] == "DRAW")
avg_duration = sum(r['turns'] for r in showdown_results) / total_matches

v10_win_rate = (v10_wins / total_matches) * 100
v8_win_rate = (v8_wins / total_matches) * 100
draw_rate = (draws / total_matches) * 100

# ─── 5. The Final Leaderboard ────────────────────────────────────────────────
print("\n👑 ─────── THE GRAND SHOWDOWN FINAL LEADERBOARD ───────")
print(f"🏁 Total Matches Conducted: {total_matches}")
print(f"⏱️  Average Match Duration:  {avg_duration:.1f} turns")
print("────────────────────────────────────────────────────────")
print(f"🚀 Optimized V10 (Production Rush): {v10_wins} Wins ({v10_win_rate:.2f}%)")
print(f"🛡️  Optimized V8 (Defensive Anchor): {v8_wins} Wins ({v8_win_rate:.2f}%)")
print(f"🤝 Standoffs / Structural Draws:   {draws} Draws ({draw_rate:.2f}%)")
print("────────────────────────────────────────────────────────")

if v10_wins > v8_wins:
    print("🏆 FINAL VERDICT: V10 'PRODUCTION RUSH' IS THE DOMINANT ARCHITECTURE!")
elif v8_wins > v10_wins:
    print("🏆 FINAL VERDICT: V8 'DEFENSIVE ANCHOR' IS THE DOMINANT ARCHITECTURE!")
else:
    print("🏆 FINAL VERDICT: THE ARCHITECTURES ARE PERFECTLY BALANCED!")

✅ Found optimized agents! V8 (Defensive) vs V10 (Aggressive).

⚔️ Initiating the Grand Showdown over 150 unseen maps...
Alternating player seating positions to ensure unbiased evaluation...

👑 ─────── THE GRAND SHOWDOWN FINAL LEADERBOARD ───────
🏁 Total Matches Conducted: 150
⏱️  Average Match Duration:  258.6 turns
────────────────────────────────────────────────────────
🚀 Optimized V10 (Production Rush): 71 Wins (47.33%)
🛡️  Optimized V8 (Defensive Anchor): 79 Wins (52.67%)
🤝 Standoffs / Structural Draws:   0 Draws (0.00%)
────────────────────────────────────────────────────────
🏆 FINAL VERDICT: V8 'DEFENSIVE ANCHOR' IS THE DOMINANT ARCHITECTURE!
